# Modul 05 · nb06 — Conversational RAG

## Masalah: RAG satu-shot tidak mengingat percakapan

Pada notebook sebelumnya (nb01–05) kita membangun pipeline RAG yang menjawab satu pertanyaan mandiri.  
Masalah muncul saat pengguna mengajukan **pertanyaan lanjutan** dalam percakapan:

| Giliran | Pertanyaan pengguna | Masalah |
|---------|---------------------|---------|
| 1 | "Berapa hari cuti tahunan?" | Aman — mandiri |
| 2 | "Kalau cuti sakit?" | **Elipsis** — "kalau" merujuk ke konteks giliran 1 |
| 3 | "Apakah perlu surat dokter?" | **Kata ganti implisit** — "itu" / "-nya" mengacu cuti sakit |
| 4 | "Bagaimana cara mengajukannya?" | **Elipsis lagi** — "-nya" mengacu pengajuan cuti |

Jika kita langsung meng-embed "Kalau cuti sakit?", hasilnya **tidak nyambung ke dokumen manapun** — embedding terlalu pendek dan ambigu tanpa konteks.  
Retrieval akan gagal atau mengembalikan dokumen yang salah.

## Solusi: memori percakapan + history-aware rewriting

1. **Memori percakapan** — simpan riwayat giliran (user ↔ asisten) agar tersedia saat menjawab pertanyaan berikutnya.
2. **History-aware rewriting** — sebelum retrieve, tulis ulang pertanyaan lanjutan menjadi pertanyaan **mandiri** (standalone) yang bisa di-embed dengan benar.

```
User (lanjutan)
      ↓
  rewrite_followup(history, follow_up)   ← menggunakan riwayat
      ↓
  pertanyaan mandiri
      ↓
  embed → retrieve → rerank
      ↓
  generate (Qwen, grounded)
      ↓
  simpan ke ConversationalMemoryManager
```

> Notebook ini dirancang untuk Google Colab dengan **GPU T4** (Qwen2.5-3B 4-bit + reranker).

In [1]:
# Install dependencies
!pip install -q "transformers<5" "sentence-transformers>=3.0" faiss-cpu accelerate bitsandbytes

import os, sys

# Clone the course repo if running on Colab without local files
REPO = "/content/navasena-gen-ml-course"
if not os.path.exists(REPO):
    !git clone https://github.com/chmdznr/navasena-gen-ml-course.git {REPO}

# Make rag_utils importable
sys.path.append(os.path.abspath(f"{REPO}/05_rag"))

from tools.rag_utils import ConversationalMemoryManager
print("Import ConversationalMemoryManager: OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.6 MB/s eta 0:00:00
Cloning into '/content/navasena-gen-ml-course'...
remote: Enumerating objects: 1344, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 1344 (delta 54), reused 68 (delta 42), pack-reused 1255 (from 1)
Receiving objects: 100% (1344/1344), 49.62 MiB | 31.33 MiB/s, done.
Resolving deltas: 100% (795/795), done.
Import ConversationalMemoryManager: OK


## Dua jenis memori + history-aware rewriting

### 1. Window memory
Menyimpan **N giliran terakhir secara verbatim**.  
Cepat dan presisi — tidak ada informasi yang hilang dalam window.  
Kelemahannya: jika percakapan panjang, konteks bisa membengkak dan melebihi batas token model.

### 2. Summary memory
Giliran yang sudah keluar dari window di-**ringkas oleh model** menjadi satu-dua kalimat.  
Ringkasan ini disertakan di awal konteks sehingga model tetap "ingat" tema percakapan tanpa harus membaca semua giliran lama.

### 3. History-aware rewriting
Sebelum retrieve, `rewrite_followup(history_context, follow_up)` meminta model untuk **mengganti kata ganti dan elipsis** dengan entitas eksplisit dari riwayat.  
Hasilnya: pertanyaan mandiri yang embedding-nya bisa cocok dengan dokumen relevan.

### Kenapa hand-roll?
Kita **tidak** menggunakan `ConversationBufferWindowMemory` atau `ConversationSummaryMemory` dari LangChain karena keduanya sudah **deprecated** di LangChain 0.3.x.  
Dengan hand-roll, kodenya **transparan, stabil, dan mudah di-debug**.  
Di akhir notebook kita tunjukkan padanan library secara jujur.

In [2]:
# ── Corpus: 12 pasal handbook karyawan (sama seperti nb05) ──────────────────
corpus = [
    "Setiap pegawai tetap memperoleh jatah dua belas hari libur berbayar dalam satu tahun, terpisah dari hari libur nasional.",
    "Pengajuan cuti tahunan dilakukan melalui portal karyawan dan wajib mendapat persetujuan atasan langsung paling lambat tiga hari sebelumnya.",
    "Cuti sakit dapat diambil hingga sepuluh hari setiap tahun dan harus disertai surat keterangan dari dokter bila lebih dari dua hari berturut-turut.",
    "Karyawan baru menjalani masa orientasi selama tiga hari kerja pertama, mencakup pengenalan budaya perusahaan dan pelatihan keselamatan.",
    "Cuti melahirkan diberikan selama tiga bulan penuh sesuai ketentuan ketenagakerjaan yang berlaku di Indonesia.",
    "Jam kerja standar adalah pukul 09.00 hingga 17.00 dari Senin sampai Jumat, dengan satu jam istirahat untuk makan siang.",
    "Karyawan dapat bekerja dari rumah maksimal dua hari dalam seminggu setelah mendapat persetujuan dari manajer tim.",
    "Gaji dibayarkan pada tanggal 25 setiap bulan melalui transfer bank ke rekening masing-masing pegawai.",
    "Perusahaan menyediakan asuransi kesehatan yang mencakup rawat inap dan rawat jalan bagi pegawai beserta satu anggota keluarga.",
    "Lembur pada hari libur dihitung dua kali lipat dari tarif normal dan harus dicatat dalam sistem absensi sebelum dikerjakan.",
    "Pelanggaran kode etik dapat berujung pada surat peringatan hingga pemutusan hubungan kerja sesuai tingkat keparahannya.",
    "Penggantian biaya perjalanan dinas diajukan paling lambat tujuh hari setelah perjalanan dengan melampirkan bukti pembayaran yang sah.",
]

# ── Embedder + FAISS index ───────────────────────────────────────────────────
import warnings, faiss, numpy as np, torch
warnings.filterwarnings("ignore", message=".*_check_is_size.*")   # bitsandbytes/torch internal, non-fatal
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
cvecs = embedder.encode(corpus, convert_to_numpy=True).astype("float32")
index = faiss.IndexFlatL2(cvecs.shape[1]); index.add(cvecs)

# ── Cross-encoder reranker (bge-reranker-v2-m3, from nb03) ──────────────────
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)

# ── Qwen2.5-3B-Instruct in 4-bit (T4-safe) ──────────────────────────────────
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
model_name = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(model_name)
gen = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map="auto")
# do_sample=False (greedy): kosongkan param sampling dari config agar tak ada warning 'flags not valid'
for _k in ("temperature", "top_p", "top_k"):
    setattr(gen.generation_config, _k, None)
print("Pipeline siap:", gen.device)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Pipeline siap: cuda:0


In [3]:
# ── Qwen helpers ─────────────────────────────────────────────────────────────

def qwen(prompt, max_new_tokens=160):
    messages = [{"role": "system", "content": "Jawab ringkas dalam Bahasa Indonesia."},
                {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(gen.device)
    out = gen.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()


def summarize_turn(old_summary, turn):                  # di-inject ke ConversationalMemoryManager
    user, assistant = turn
    return qwen(f"Ringkasan saat ini: {old_summary or '(kosong)'}\n"
                f"Perbarui ringkasan singkat dengan menambahkan giliran ini:\n"
                f"User: {user}\nAsisten: {assistant}\n\nRingkasan baru (1-2 kalimat):", 80)


def rewrite_followup(history_context, follow_up):       # history-aware query rewriting
    if not history_context.strip():
        return follow_up
    return qwen(f"Riwayat percakapan:\n{history_context}\n\n"
                f"Pertanyaan lanjutan: {follow_up}\n\n"
                f"Tulis ulang HANYA pertanyaan lanjutan menjadi SATU pertanyaan mandiri yang singkat.\n"
                f"Aturan:\n"
                f"1. PERTAHANKAN topik pertanyaan lanjutan. Jika ia menyebut 'cuti sakit', "
                f"hasilnya harus tentang cuti sakit — JANGAN diganti topik lama dari riwayat.\n"
                f"2. Pakai riwayat hanya untuk mengisi yang hilang (kata ganti 'itu'/'-nya' atau predikat).\n"
                f"3. Maksimal satu kalimat tanya.\n"
                f"Keluarkan HANYA pertanyaannya:", 60)


print("Helpers didefinisikan: qwen, summarize_turn, rewrite_followup")

Helpers didefinisikan: qwen, summarize_turn, rewrite_followup


In [4]:
# ── Retriever dengan reranking ────────────────────────────────────────────────

def retrieve_rerank(query, k_over=12, k_top=3):
    qv = embedder.encode([query], convert_to_numpy=True).astype("float32")
    _, idx = index.search(qv, k_over); cand = idx[0].tolist()
    scores = reranker.predict([(query, corpus[c]) for c in cand]).tolist()
    order = sorted(range(len(cand)), key=lambda i: scores[i], reverse=True)[:k_top]
    return [corpus[cand[i]] for i in order]


# window=2 agar summary aktif lebih cepat dalam demo
memory = ConversationalMemoryManager(summarize_fn=summarize_turn, window=2)


def chat_turn(follow_up):
    standalone = rewrite_followup(memory.context(), follow_up)
    chunks = retrieve_rerank(standalone)
    ctx = "\n".join(f"- {c}" for c in chunks)
    answer = qwen(f"Konteks:\n{ctx}\n\nPertanyaan: {standalone}\n\n"
                  f"Jawab HANYA berdasarkan konteks. Bila konteks tidak memuat jawabannya, "
                  f"katakan terus terang bahwa informasinya tidak tersedia.", max_new_tokens=256)
    memory.add_turn(follow_up, answer)
    return standalone, answer


print("Retriever + chat_turn siap")

Retriever + chat_turn siap


## Alur satu giliran percakapan

```
pertanyaan pengguna (mungkin elipsis / kata ganti)
        │
        ▼
 rewrite_followup(memory.context(), follow_up)
        │   ← menggunakan riwayat window + ringkasan
        ▼
 pertanyaan MANDIRI yang bisa di-embed
        │
        ▼
 retrieve_rerank(standalone)   ← embed → FAISS → rerank bge-reranker-v2-m3
        │
        ▼
 top-3 chunks relevan
        │
        ▼
 qwen(konteks + pertanyaan)    ← generate grounded (do_sample=False)
        │
        ▼
 jawaban
        │
        ▼
 memory.add_turn(follow_up, answer)   ← window evicts lama → summarize_turn
```

Kunci: **retrieve memakai pertanyaan mandiri**, bukan pertanyaan lanjutan mentah —  
sehingga embedding cocok dengan dokumen korpus.

## Demo percakapan — 4 giliran

Giliran 2–4 **tidak bisa di-retrieve langsung** tanpa riwayat.  
Perhatikan kolom `(rewrite -> ...)` — di situlah history-aware rewriting bekerja.

| # | Pertanyaan mentah | Jenis ambiguitas |
|---|-------------------|------------------|
| 1 | Berapa hari cuti tahunan untuk pegawai tetap? | Mandiri ✓ |
| 2 | Kalau cuti sakit? | Elipsis — "kalau" = "berapa hari" |
| 3 | Apakah perlu surat dokter? | Kata ganti implisit — merujuk cuti sakit |
| 4 | Bagaimana cara mengajukannya? | "-nya" = pengajuan cuti |

> Catatan jujur: rewriting di sini memakai model **kecil** (Qwen2.5-3B), jadi hasilnya **bisa berbeda tiap run** dan kadang **keliru** — misalnya mempertahankan topik lama ("cuti tahunan") alih-alih beralih ke topik baru ("cuti sakit"). Idealnya pertanyaan hasil rewrite jadi lebih spesifik sehingga retrieval menemukan dokumen yang tepat. Perhatikan kolom `(rewrite -> ...)` di output dan bandingkan apakah topiknya sudah benar. Di **cell 15** nanti, model **cloud** yang lebih besar menyelesaikan kasus serupa dengan lebih bersih.

In [5]:
# ── Scripted demo: 4 giliran percakapan ──────────────────────────────────────

memory.clear()

turns = [
    "Berapa hari cuti tahunan untuk pegawai tetap?",
    "Kalau cuti sakit?",                  # target rewrite (approx): "Berapa hari cuti sakit?"
    "Apakah perlu surat dokter?",         # target rewrite (approx): "Apakah cuti sakit perlu surat keterangan dokter?"
    "Bagaimana cara mengajukannya?",      # target rewrite (approx): "Bagaimana cara mengajukan cuti?"
]

for t in turns:
    standalone, answer = chat_turn(t)
    print(f"User    : {t}")
    print(f"  (rewrite -> {standalone})")
    print(f"Asisten : {answer}\n{'─'*60}")

User    : Berapa hari cuti tahunan untuk pegawai tetap?
  (rewrite -> Berapa hari cuti tahunan untuk pegawai tetap?)
Asisten : Pegawai tetap memperoleh jatah cuti tahunan sebesar 12 hari.
────────────────────────────────────────────────────────────
User    : Kalau cuti sakit?
  (rewrite -> Kalau cuti sakit? Bagaimana jatah cuti tahunan pegawai tetap?)
Asisten : Jatah cuti tahunan pegawai tetap memerlukan pengajuan melalui portal karyawan seperti yang ditentukan dalam konteks tersebut. Tidak ada pengecualian untuk cuti sakit yang mempengaruhi jatah cuti tahunan. Jadi, pegawai tetap memperoleh jatah dua belas hari libur berbayar dalam satu tahun, terpisah dari hari libur nasional, tanpa mengganggu jatah cuti tahunan tersebut.
────────────────────────────────────────────────────────────
User    : Apakah perlu surat dokter?
  (rewrite -> Apakah perlu surat dari dokter untuk cuti sakit bagi pegawai tetap itu?)
Asisten : Ya, berdasarkan konteks yang diberikan, untuk cuti sakit, pegawai tetap

## Analisis hasil

### Mengapa rewriting wajib?

Coba bandingkan embedding dua pertanyaan ini secara intuitif:

- **Tanpa rewrite**: `"Kalau cuti sakit?"` → embedding pendek, ambigu → retriever bisa mengembalikan dokumen tentang cuti tahunan atau jam kerja, bukan cuti sakit
- **Target rewrite (ideal)**: `"Berapa hari cuti sakit yang diperbolehkan?"` → embedding spesifik → retriever menemukan *"Cuti sakit dapat diambil hingga sepuluh hari..."*

> ⚠️ **Realita model kecil.** Hasil di atas adalah **target ideal**, bukan jaminan. Pada run nyata, Qwen2.5-3B kadang **menyeret topik lama**: "Kalau cuti sakit?" bisa tertulis ulang jadi "...berapa hari **cuti tahunan**..." sehingga jawabannya melenceng. Periksa output cell 8 — kalau ini terjadi pada run-mu, itulah batas model kecil, dan justru bahan pelajaran. Model cloud yang lebih besar (**cell 15**) menyelesaikan elipsis yang sama dengan benar.

### Pola elipsis yang ditangani

| Pola | Contoh | Resolusi ideal |
|------|--------|----------------|
| Elipsis predikat | "Kalau cuti sakit?" | Tambahkan predikat, **pertahankan topik** ("berapa hari [cuti sakit]") |
| Kata ganti "-nya" | "mengajukannya" | Ganti dengan objek eksplisit ("mengajukan cuti") |
| Referensi implisit | "Apakah perlu...?" | Tambahkan subjek dari giliran sebelumnya |

### Itulah inti conversational RAG

Bukan sekadar "tambahkan chat history ke prompt" — tapi **tulis ulang pertanyaan** menjadi mandiri
**sebelum** embedding/retrieval, sehingga semantic search bekerja dengan benar.

> Jawaban model juga bisa **berhalusinasi** (mengarang fakta di luar korpus), apalagi pada model 3B. Selalu periksa jawaban terhadap dokumen sumber — di nb07 kita tambahkan **sitasi** agar setiap klaim bisa ditelusuri.

Setelah 4 giliran dengan `window=2`, giliran pertama sudah keluar dari window dan diringkas. Perhatikan field **`summary`** / `has_summary` pada output `memory.stats()` — itu bukti path *summary memory* benar-benar aktif.

In [6]:
# ── Stats memori + export JSON ────────────────────────────────────────────────
import json

print("Memori:", memory.stats())                      # has_summary True berarti ada giliran lama yang sudah diringkas
convo = [{"user": u, "assistant": a} for u, a in memory.turns]
json.dump({"summary": memory.summary, "window_turns": convo},
          open("/content/percakapan.json", "w"), ensure_ascii=False, indent=2)
print("Percakapan diekspor ke /content/percakapan.json")
# Catatan: /content/ di Colab bersifat sementara — unduh file sebelum sesi berakhir.

Memori: {'turns_kept': 2, 'has_summary': True}
Percakapan diekspor ke /content/percakapan.json


> Sel ini interaktif: setelah mengetik di kotak **Anda:**, tekan **Enter**. Qwen butuh beberapa detik untuk menjawab (tak ada indikator), jadi diam sebentar itu normal. Ajukan pertanyaan tentang dokumen HR (mis. cuti, jam kerja, gaji), bukan sapaan seperti 'halo'.

In [7]:
# Loop ngobrol interaktif. Setelah mengetik, TEKAN ENTER untuk mengirim.
# Perintah: 'quit' (berhenti), 'stats' (lihat memori), 'clear' (kosongkan memori).
memory.clear()                                   # mulai sesi dari memori kosong (hindari sisa state cell 8)
print("=== Mode ngobrol ===")
print("Coba tanya tentang kebijakan HR, mis: 'Berapa hari cuti tahunan?' lalu Enter.")
print("Ketik 'quit' untuk berhenti.\n")
while True:
    msg = input("Anda: ").strip()
    if not msg:                                  # input kosong -> minta lagi
        continue
    if msg.lower() == "quit":
        print("Sampai jumpa!"); break
    if msg.lower() == "stats":
        print("  ", memory.stats()); continue
    if msg.lower() == "clear":
        memory.clear(); print("  (memori dikosongkan)"); continue
    print("  ⏳ Qwen sedang berpikir (~beberapa detik, tanpa loading bar)...")
    standalone, answer = chat_turn(msg)
    if standalone != msg:
        print(f"  (pertanyaan ditulis ulang -> {standalone})")
    print(f"  Asisten: {answer or '(jawaban kosong — coba pertanyaan tentang dokumen HR)'}\n")

=== Mode ngobrol ===
Coba tanya tentang kebijakan HR, mis: 'Berapa hari cuti tahunan?' lalu Enter.
Ketik 'quit' untuk berhenti.

Anda: tadi bicara apa?
  ⏳ Qwen sedang berpikir (~beberapa detik, tanpa loading bar)...
  Asisten: Informasi yang Anda sampaikan berkaitan dengan aturan dan kebijakan perusahaan, seperti jam kerja, cuti, dan aturan cuti sakit. Tapi tidak ada informasi tentang apa yang dibicarakan sebelumnya.

Anda: berapa hari cuti tahunan?
  ⏳ Qwen sedang berpikir (~beberapa detik, tanpa loading bar)...
  (pertanyaan ditulis ulang -> Berapa hari cuti tahunan itu?)
  Asisten: Dalam konteks yang diberikan, informasi tentang berapa hari cuti tahunan tidak disebutkan. Jadi, jawabannya adalah:

Informasi ini tidak tersedia dalam konteks yang diberikan.

Anda: quit
Sampai jumpa!


## Referensi runnable: `create_history_aware_retriever` ala LangChain

Selama ini kita **hand-roll** rewriting + memori. Library standar **LangChain** punya komponen siap pakai bernama `create_history_aware_retriever` — tugasnya persis seperti `rewrite_followup` kita: menulis ulang pertanyaan lanjutan menjadi mandiri, lalu retrieve.

Yang berbeda di sel ini: kita **tidak memuat LLM lokal lagi** (GPU sudah penuh oleh Qwen). Kita pinjam **LLM cloud NVIDIA NIM** — infrastruktur yang sama dengan judge di nb04 — sehingga langkah rewriting jalan **tanpa GPU tambahan**. Embedder yang sudah dimuat (`embedder`) dipakai ulang lewat adapter kecil, jadi **tidak perlu** install `sentence-transformers` / `langchain-huggingface` (yang bisa menarik `transformers` 5.x dan menabrak Qwen 4-bit).

> Butuh `NVIDIA_API_KEY` (gratis di [build.nvidia.com](https://build.nvidia.com)). Simpan di **Colab Secrets** (ikon 🔑 di panel kiri) dengan nama `NVIDIA_API_KEY`. Bagian ini opsional — kalau tidak ada key, sel otomatis dilewati.

In [8]:
# ── Referensi runnable: create_history_aware_retriever (LLM = NVIDIA NIM cloud, tanpa GPU) ──
# Install RINGAN — hanya paket LangChain. JANGAN tambah sentence-transformers / langchain-huggingface
# di sini: keduanya bisa menarik transformers 5.x dan menabrak Qwen 4-bit yang sudah dimuat.
!pip install -q "langchain-classic==1.0.8" "langchain-community==0.4.2" "langchain-nvidia-ai-endpoints==1.4.1"

import os, getpass

# NVIDIA_API_KEY dari Colab Secrets (ikon kunci di kiri) — JANGAN hardcode di sel.
try:
    from google.colab import userdata
    os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY") or ""
except Exception:
    if not os.environ.get("NVIDIA_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA_API_KEY (Enter untuk lewati): ")

if not os.environ.get("NVIDIA_API_KEY"):
    print("Tidak ada NVIDIA_API_KEY — demo LangChain dilewati (bagian ini opsional/referensi).")
else:
    from langchain_core.embeddings import Embeddings
    from langchain_community.vectorstores import FAISS as LCFAISS
    from langchain_nvidia_ai_endpoints import ChatNVIDIA
    from langchain_classic.chains import create_history_aware_retriever
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.messages import HumanMessage, AIMessage
    from langchain_core.output_parsers import StrOutputParser

    # Adapter: pakai ulang `embedder` (SentenceTransformer) yang sudah dimuat di atas,
    # supaya tidak perlu install langchain-huggingface / sentence-transformers lagi.
    class STAdapter(Embeddings):
        def __init__(self, model): self.model = model
        def embed_documents(self, texts):
            return self.model.encode(texts, convert_to_numpy=True).tolist()
        def embed_query(self, text):
            return self.model.encode([text], convert_to_numpy=True)[0].tolist()

    lc_retriever = LCFAISS.from_texts(corpus, STAdapter(embedder)).as_retriever(search_kwargs={"k": 3})
    lc_llm = ChatNVIDIA(model="meta/llama-3.3-70b-instruct", temperature=0, max_completion_tokens=128)

    lc_prompt = ChatPromptTemplate.from_messages([
        ("system", "Berdasarkan riwayat percakapan, tulis ulang pertanyaan terakhir pengguna "
                   "menjadi pertanyaan yang berdiri sendiri (mandiri). Jangan dijawab, cukup tulis ulang."),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])

    chat_history = [
        HumanMessage(content="Berapa hari cuti tahunan yang saya dapat?"),
        AIMessage(content="Setiap pegawai tetap memperoleh dua belas hari cuti tahunan berbayar."),
    ]
    followup = "Kalau yang sakit bagaimana?"      # tanpa riwayat, kalimat ini ambigu

    try:
        # (a) Lihat hasil rewriting-nya secara eksplisit
        rewrite_chain = lc_prompt | lc_llm | StrOutputParser()
        standalone = rewrite_chain.invoke({"input": followup, "chat_history": chat_history}).strip()
        print(f"Pertanyaan asli : {followup}")
        print(f"Ditulis ulang   : {standalone}\n")

        # (b) create_history_aware_retriever = rewriting + retrieval dalam satu komponen
        har = create_history_aware_retriever(lc_llm, lc_retriever, lc_prompt)
        docs = har.invoke({"input": followup, "chat_history": chat_history})
        print(f"Dokumen teratas ({len(docs)} diambil):")
        for i, d in enumerate(docs, 1):
            print(f"  [{i}] {d.page_content[:70]}...")
    except Exception as e:
        print(f"Gagal memanggil NVIDIA NIM (cek API key / kuota): {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_1780/1241456209.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS as LCFAISS


Pertanyaan asli : Kalau yang sakit bagaimana?
Ditulis ulang   : Berapa hari cuti sakit yang saya dapat?

Dokumen teratas (3 diambil):
  [1] Cuti sakit dapat diambil hingga sepuluh hari setiap tahun dan harus di...
  [2] Lembur pada hari libur dihitung dua kali lipat dari tarif normal dan h...
  [3] Jam kerja standar adalah pukul 09.00 hingga 17.00 dari Senin sampai Ju...


**Apa yang baru saja terjadi?** `create_history_aware_retriever` menerima pertanyaan ambigu *"Kalau yang sakit bagaimana?"* + riwayat percakapan, lalu menulis ulang menjadi *"Berapa hari cuti sakit yang saya dapat?"* — persis tugas `rewrite_followup` kita — sebelum retrieval. Hasilnya pasal **cuti sakit** muncul di urutan teratas. Bandingkan dengan demo lokal (cell 8): model **3B** kita kadang menyeret topik lama saat menulis ulang, sedangkan model **cloud** yang lebih besar di sini menyelesaikan elipsis *"yang sakit"* → *"cuti sakit"* dengan bersih. **Kualitas rewriting naik seiring kapasitas model** — itu sebabnya langkah rewriting sering dilimpahkan ke model yang lebih kuat.

Contoh output (model cloud bisa sedikit berbeda pilihan katanya):

```
Pertanyaan asli : Kalau yang sakit bagaimana?
Ditulis ulang   : Berapa hari cuti sakit yang saya dapat?

Dokumen teratas (3 diambil):
  [1] Cuti sakit dapat diambil hingga sepuluh hari setiap tahun da...
```

Adapun **penyimpanan memori** di LangChain berpindah paradigma ~3x dalam 2 tahun:

- `ConversationBufferWindowMemory` / `ConversationSummaryMemory` → **deprecated** (0.3.x)
- `RunnableWithMessageHistory` → **juga deprecated** (core 1.3.3)
- Sekarang: **LangGraph** (`InMemorySaver` + `MessagesState` + `trim_messages`)

Karena API-nya sering berganti, kita **hand-roll** inti window+summary (`ConversationalMemoryManager`) yang sederhana, stabil, dan transparan — sambil tetap paham mekanismenya. **LangChain** unggul untuk integrasi cepat; **hand-roll** unggul untuk kontrol penuh dan kestabilan jangka panjang.

## Ringkasan & jembatan ke nb07

### Yang sudah kita bangun di nb06

| Komponen | Teknik | File |
|----------|--------|------|
| Window memory | Verbatim N giliran terakhir | `rag_utils.py` |
| Summary memory | Qwen meringkas giliran lama | `rag_utils.py` |
| History-aware rewriting | `rewrite_followup` — resolusi elipsis + kata ganti | nb06 |
| Retrieve + rerank | bi-encoder → bge-reranker-v2-m3 | nb06 (dari nb03) |
| Grounded generation | `apply_chat_template` + `do_sample=False` | nb06 |

### Jembatan ke nb07 — Capstone: Ask-My-Document

nb07 menyatukan **semua** yang telah kita pelajari:

1. **Ingest + chunk** (nb02) — upload PDF pengguna, chunk dengan strategi terbaik
2. **Retrieve + rerank** (nb03) — FAISS + bge-reranker-v2-m3
3. **Conversational** (nb06) — memori multi-turn + history-aware rewriting
4. **Jawaban bersitasi** — setiap klaim dilengkapi nomor halaman/pasal sumber

Pengguna cukup upload satu PDF → tanya apa saja → sistem mengingat konteks percakapan.